In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize.treebank import TreebankWordDetokenizer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
CSVPATH = 'Coachella-2015-2-DFE.csv' #Put your path
df = pd.read_csv(CSVPATH, encoding='cp1251')
df.head()

,coachella_sentiment,coachella_yn,name,retweet_count,text,tweet_coord,tweet_created,tweet_id,tweet_location,user_timezone
0,positive,yes,kokombil,0,#Coachella2015 tickets selling out in less tha...,"[0.0, 0.0]",1/7/15 15:02,5.529630e+17,NaN,Quito
1,positive,yes,MisssTaraaa10,2,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ЩУ...,NaN,1/7/15 15:02,5.529630e+17,united states,NaN
2,positive,yes,NMcCracken805,0,#Coachella2015 #VIP passes secured! See you th...,NaN,1/7/15 15:01,5.529630e+17,"Costa Mesa, CA",NaN
3,positive,yes,wxpnfm,1,Philly‰ЫЄs @warondrugsjams will play #Coachell...,NaN,1/7/15 15:01,5.529630e+17,"Philadelphia, PA and Worldwide",Quito
4,positive,yes,Caesears,0,If briana and her mom out to #Coachella2015 i...,NaN,1/7/15 15:00,5.529630e+17,NaN,NaN


In [ ]:
#Extract all the hashtags
def hashtagextractor(strinput):
  pattern = r"#\w+"
  tag = re.findall(pattern, strinput)
  return ",".join(tag)
#Extract all the emails
def emailextractor(strinput):
  pattern = r"\S{1,64}@[a-zA-Z]+\.[a-z]+"
  emails = re.findall(pattern, strinput)
  return ",".join(emails)
#Remove Usernames
def remove_usernames(strinput):
  pattern = r"@\w+"
  removed = re.sub(pattern, '',strinput)
  return removed
#Remove Links
def remove_links(strinput):
  pattern = r"\bhttps?://[^\s[\]{}<>|]+"
  removed = re.sub(pattern, '',strinput)
  return removed
#Remove any symbols that are not part of the ASCII
def remove_non_ascii_symbols(strinput):
  pattern = r"[^\x00-\x7F]+"
  removed = re.sub(pattern, '',strinput)
  return removed
#Convert all the letters to lower case
def to_lower(strinput):
  return strinput.lower()
#Remove any common words that do not add much meaning
def remove_stop_words(strinput):
  tokens = nltk.word_tokenize(strinput)
  stopwords = nltk.corpus.stopwords.words("english")
  filtered_tokens = [token for token in tokens if token.lower() not in stopwords]
  detokenizer = TreebankWordDetokenizer()
  text = detokenizer.detokenize(filtered_tokens)
  return text
#Remove any numbers
def remove_digits(strinput):
  pattern = r"\d+"
  removed = re.sub(pattern, '',strinput)
  return removed
#Remove any punctuation marks or other special characters
def remove_special_characters(strinput):
  pattern = r"[^\w\s_]+|_"
  removed = re.sub(pattern, '',strinput)
  return removed

In [ ]:
df['hashtags'] = df['text'].apply(hashtagextractor)
df['emails'] = df['text'].apply(emailextractor)

df['newtext'] = (
    df['text'].apply(remove_usernames)
    .apply(remove_links)
    .apply(remove_non_ascii_symbols)
    .apply(to_lower)
    .apply(remove_stop_words)
    .apply(remove_digits)
    .apply(remove_special_characters)
)
df[['hashtags','emails']].head()

,hashtags,emails
0,#Coachella2015,
1,#Coachella2015,
2,"#Coachella2015,#VIP",
3,"#Coachella2015,#GovBall2015",
4,#Coachella2015,


In [ ]:
df[['text','newtext']].head()
df.to_csv('cleaned.csv', index=False)